In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

def preprocess_data(df, numeric_columns=None):
    """
    데이터 전처리 함수: 결측값과 이상치를 처리
    
    Parameters:
    df: pandas DataFrame - 처리할 데이터프레임
    numeric_columns: list - 수치형 컬럼 리스트 (None인 경우 자동 감지)
    """
    
    # 원본 데이터 복사
    df_processed = df.copy()
    
    print("=== 데이터 전처리 시작 ===")
    print(f"원본 데이터 크기: {df_processed.shape}")
    
    # 1. 결측치 분석
    print("\n=== 결측치 분석 ===")
    missing_stats = df_processed.isnull().sum()
    missing_stats = missing_stats[missing_stats > 0]
    if len(missing_stats) > 0:
        print("\n컬럼별 결측치 개수:")
        print(missing_stats)
        
        # 결측치 시각화
        plt.figure(figsize=(10, 6))
        sns.heatmap(df_processed.isnull(), yticklabels=False, cbar=True, cmap='viridis')
        plt.title('Missing Values Heatmap')
        plt.show()
    else:
        print("결측치가 없습니다.")
    
    # 2. 결측치 처리
    print("\n=== 결측치 처리 ===")
    
    # 수치형 컬럼 자동 감지 (지정되지 않은 경우)
    if numeric_columns is None:
        numeric_columns = df_processed.select_dtypes(include=[np.number]).columns
    
    # 범주형 컬럼
    categorical_columns = df_processed.select_dtypes(include=['object']).columns
    
    # 수치형 컬럼 결측치 처리 (중앙값으로 대체)
    for col in numeric_columns:
        if df_processed[col].isnull().sum() > 0:
            median_value = df_processed[col].median()
            df_processed[col].fillna(median_value, inplace=True)
            print(f"{col}: {df_processed[col].isnull().sum()}개 결측치를 중앙값 {median_value:.2f}로 대체")
    
    # 범주형 컬럼 결측치 처리 (최빈값으로 대체)
    for col in categorical_columns:
        if df_processed[col].isnull().sum() > 0:
            mode_value = df_processed[col].mode()[0]
            df_processed[col].fillna(mode_value, inplace=True)
            print(f"{col}: {df_processed[col].isnull().sum()}개 결측치를 최빈값 '{mode_value}'로 대체")
    
    # 3. 이상치 분석 및 처리
    print("\n=== 이상치 분석 및 처리 ===")
    
    def detect_outliers(df, column):
        """
        IQR 방식을 사용한 이상치 감지
        """
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)][column]
        return outliers, lower_bound, upper_bound
    
    # 수치형 컬럼의 이상치 처리
    for col in numeric_columns:
        # 이상치 감지
        outliers, lower_bound, upper_bound = detect_outliers(df_processed, col)
        
        if len(outliers) > 0:
            print(f"\n{col} 컬럼 이상치 분석:")
            print(f"이상치 개수: {len(outliers)}")
            print(f"하한값: {lower_bound:.2f}")
            print(f"상한값: {upper_bound:.2f}")
            
            # 박스플롯으로 이상치 시각화
            plt.figure(figsize=(10, 6))
            sns.boxplot(x=df_processed[col])
            plt.title(f'Boxplot of {col}')
            plt.show()
            
            # 이상치 처리 (상한/하한값으로 대체)
            df_processed.loc[df_processed[col] < lower_bound, col] = lower_bound
            df_processed.loc[df_processed[col] > upper_bound, col] = upper_bound
            print("이상치를 상한/하한값으로 대체 완료")
            
            # 처리 후 박스플롯
            plt.figure(figsize=(10, 6))
            sns.boxplot(x=df_processed[col])
            plt.title(f'Boxplot of {col} (After Outlier Treatment)')
            plt.show()
    
    # 4. 처리 결과 요약
    print("\n=== 전처리 완료 ===")
    print(f"처리된 데이터 크기: {df_processed.shape}")
    
    # 기본 통계량 출력
    print("\n처리된 데이터 기본 통계량:")
    print(df_processed[numeric_columns].describe())
    
    return df_processed

# 사용 예시
if __name__ == "__main__":
    # 예시 데이터 생성
    np.random.seed(42)
    data = {
        'age': [25, 30, np.nan, 40, 35, 28, 45, 50, 1000, 32],
        'salary': [50000, 60000, 75000, np.nan, 65000, 55000, 80000, 85000, 90000, 1000000],
        'department': ['IT', 'HR', np.nan, 'IT', 'Finance', 'HR', 'IT', np.nan, 'Finance', 'IT']
    }
    df = pd.DataFrame(data)
    
    # 전처리 함수 실행
    processed_df = preprocess_data(df, numeric_columns=['age', 'salary'])